# Advanced 11 — Information Inequalities and Cramer-Rao Bounds

Practice notebook for the **"Information Inequalities and Cramer-Rao Bounds"** section of the stats course PDF.

## What you will learn

This notebook recaps the theory from the PDF section, then **verifies it with Python**.
Each part ends with a **"Your turn"** prompt; the full **Problem Set** at the end has
**click-to-reveal solutions**.

## How to use

1. **Read** each markdown cell, then **run** the code beneath it (`Shift+Enter`).
2. **Change parameters** and re-run — statistics is about *relationships*, not memorized formulas.
3. LaTeX uses \( \) for inline math and \[ \] / $$ $$ for display math (KaTeX-friendly).

Let's begin.


---
## Setup — run this first

NumPy for numerics, SciPy for exact distributions, Matplotlib for plots.


In [ ]:
# If anything is missing, uncomment and run once:
# !pip install numpy scipy matplotlib

import numpy as np
import matplotlib.pyplot as plt
import scipy
from scipy import stats

np.random.seed(42)
rng = np.random.default_rng(42)
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True

print("Ready. NumPy", np.__version__, "| SciPy", scipy.__version__)


---
## Part 1 — The score and Fisher information

For a parametric family \( \{f(x;\theta)\} \) (dominated by a common measure),
the **score** is

\[
U_\theta(X) = \frac{\partial}{\partial\theta}\log f(X;\theta),
\]

assuming differentiability under the integral sign.

The **Fisher information** in one observation is

$$
I(\theta) = E_\theta\!\left[U_\theta(X)^2\right]
= -E_\theta\!\left[\frac{\partial^2}{\partial\theta^2}\log f(X;\theta)\right],
$$

when the interchange of derivative and expectation is valid. The two forms agree
because \( E_\theta[U_\theta(X)]=0 \) (differentiate
\( \int f(x;\theta)\,dx = 1 \) once), and
\( \mathrm{Var}(U_\theta)=E[U_\theta^2] - E[U_\theta]^2 = E[U_\theta^2] \).

For i.i.d. \( X_1,\dots,X_n \), the information in the sample is
\( I_n(\theta) = n\, I(\theta) \).

We build a small numerical workbench to evaluate both forms — the variance of the
score and the expected second derivative of the log-likelihood — on a simulated
sample, and check they agree.


In [ ]:
def loglik_and_score_family(family, theta, x):
    """Return (loglik(theta; x), score(theta; x), d2 loglik / dtheta^2) per observation.

    Families implemented: 'bernoulli', 'poisson', 'normal_mean' (sigma2 known).
    Vectorized over x; theta is a scalar.
    """
    eps = 1e-300
    if family == "bernoulli":
        p = np.clip(theta, eps, 1 - eps)
        ll = x * np.log(p) + (1 - x) * np.log(1 - p)
        U = x / p - (1 - x) / (1 - p)
        d2 = -(x / p**2 + (1 - x) / (1 - p)**2)
    elif family == "poisson":
        lam = theta
        from scipy.special import gammaln
        ll = -lam + x * np.log(lam + eps) - gammaln(x + 1)
        U = -1.0 + x / lam
        d2 = -x / lam**2
    elif family == "normal_mean":
        mu = theta
        sigma2 = 1.0  # known sigma2 = 1 in this demo
        ll = -0.5 * np.log(2 * np.pi * sigma2) - (x - mu)**2 / (2 * sigma2)
        U = (x - mu) / sigma2
        d2 = -1.0 / sigma2 * np.ones_like(x)
    else:
        raise ValueError(family)
    return ll, U, d2

# Sanity check on Bernoulli(p): I(p) = 1 / (p (1-p)).
rng = np.random.default_rng(0)
p_true = 0.3
x = rng.binomial(1, p_true, size=2_000_000)
_, U, d2 = loglik_and_score_family("bernoulli", p_true, x)
I_var_score = np.mean(U**2)          # E[U^2]
I_neg_d2    = -np.mean(d2)           # -E[d2 log f / dtheta^2]
print(f"Bernoulli p={p_true}:")
print(f"  I(p) via Var(score)        = {I_var_score:.6f}")
print(f"  I(p) via -E[d2 log f]      = {I_neg_d2:.6f}")
print(f"  theory 1/(p(1-p))          = {1/(p_true*(1-p_true)):.6f}")

# Poisson(lam): I(lam) = 1/lam.
lam_true = 2.0
x = rng.poisson(lam_true, size=2_000_000)
_, U, d2 = loglik_and_score_family("poisson", lam_true, x)
print(f"\nPoisson lam={lam_true}:")
print(f"  I(lam) via Var(score)      = {np.mean(U**2):.6f}")
print(f"  I(lam) via -E[d2 log f]    = {-np.mean(d2):.6f}")
print(f"  theory 1/lam               = {1/lam_true:.6f}")

# N(mu, 1): I(mu) = 1.
mu_true = 1.5
x = rng.normal(mu_true, 1.0, size=2_000_000)
_, U, d2 = loglik_and_score_family("normal_mean", mu_true, x)
print(f"\nNormal mu={mu_true}, sigma2=1:")
print(f"  I(mu) via Var(score)       = {np.mean(U**2):.6f}")
print(f"  I(mu) via -E[d2 log f]     = {-np.mean(d2):.6f}")
print(f"  theory 1/sigma2             = {1.0:.6f}")


---
## Part 2 — Numerical Fisher information via finite differences

The two analytic forms for \( I(\theta) \) are expectations under the true model.
In practice we estimate them from data using the **observed information**

$$
\widehat{I}(\theta) = -\frac{1}{n}\sum_{i=1}^n \frac{\partial^2}{\partial\theta^2}\log f(X_i;\theta),
$$

or the score variance. We compute the second derivative by central finite
differences and the score by a one-sided difference, then compare all three
estimators of \( I(\theta) \) for the Poisson family.

For \( X_i \sim \mathrm{Pois}(\lambda) \),
\( \log f(x;\lambda) = -\lambda + x\log\lambda - \log(x!) \), so

$$
U_\lambda(x) = -1 + \frac{x}{\lambda}, \qquad
\frac{\partial^2 \log f}{\partial\lambda^2} = -\frac{x}{\lambda^2},
$$

and \( I(\lambda) = E[X]/\lambda^2 = \lambda/\lambda^2 = 1/\lambda \). We
confirm the finite-difference versions match the closed forms on a simulated
sample.


In [ ]:
def numerical_score_d2(family, theta, x, h=1e-5):
    """Central-difference score and second derivative of log f at theta, per observation."""
    ll_plus  = loglik_and_score_family(family, theta + h, x)[0]
    ll_minus = loglik_and_score_family(family, theta - h, x)[0]
    ll_0     = loglik_and_score_family(family, theta,      x)[0]
    U_num  = (ll_plus - ll_minus) / (2 * h)
    d2_num = (ll_plus - 2 * ll_0 + ll_minus) / (h * h)
    return U_num, d2_num

rng = np.random.default_rng(7)
lam_true = 2.5
n = 500
x = rng.poisson(lam_true, size=n)

# Observed information at the true parameter
U_ana, d2_ana = loglik_and_score_family("poisson", lam_true, x)[1:]
U_num, d2_num = numerical_score_d2("poisson", lam_true, x, h=1e-5)

print(f"Poisson lam={lam_true}, n={n}")
print(f"  I_hat via analytic score variance : {np.mean(U_ana**2):.6f}")
print(f"  I_hat via numeric score variance  : {np.mean(U_num**2):.6f}")
print(f"  I_hat via analytic -mean(d2)      : {-np.mean(d2_ana):.6f}")
print(f"  I_hat via numeric -mean(d2)       : {-np.mean(d2_num):.6f}")
print(f"  theory 1/lam                       : {1/lam_true:.6f}")

# Scan lambda and compare empirical observed information to theory 1/lam
lams = np.linspace(0.5, 6.0, 40)
I_obs = []
for lam in lams:
    d2 = loglik_and_score_family("poisson", lam, x)[2]
    I_obs.append(-np.mean(d2))

plt.plot(lams, 1.0 / lams, "k-", lw=2, label=r"theory $I(\lambda)=1/\lambda$")
plt.plot(lams, I_obs, "o-", ms=4, label=r"observed info $-\overline{\ell''(\lambda)}$")
plt.xlabel(r"$\lambda$"); plt.ylabel("information")
plt.title(f"Observed vs expected Fisher information (Poisson, n={n})")
plt.legend(); plt.show()

print("The empirical observed information tracks 1/lambda across the grid, with")
print("small-sample noise vanishing as n grows (try increasing n).")


---
## Part 3 — Cramer-Rao lower bound

**Cramer-Rao inequality.** Let \( T(X) \) be an unbiased estimator of
\( \phi(\theta) \) with finite variance. Under regularity conditions,

$$
\mathrm{Var}_\theta(T) \geq \frac{(\phi'(\theta))^2}{I(\theta)}.
$$

For i.i.d. samples, replace \( I(\theta) \) by \( n I(\theta) \), i.e.

$$
\mathrm{Var}_\theta(T) \geq \frac{(\phi'(\theta))^2}{n\, I(\theta)}.
$$

Estimators attaining the bound for all \( \theta \) are called **efficient**.

**Normal mean (from the PDF).** Let
\( X_1,\dots,X_n \stackrel{iid}{\sim} N(\mu,\sigma^2) \) with known
\( \sigma^2 \), and estimate \( \phi(\mu)=\mu \). The log-likelihood for one
observation is

$$
\log f(x;\mu) = -\tfrac{1}{2}\log(2\pi\sigma^2) - \frac{(x-\mu)^2}{2\sigma^2},
\qquad U_\mu(X) = \frac{X-\mu}{\sigma^2},
$$

so \( I(\mu)=1/\sigma^2 \), \( I_n(\mu)=n/\sigma^2 \), and the CRLB for an
unbiased estimator of \( \mu \) is \( \sigma^2/n \). The sample mean
\( \bar X \) has \( \mathrm{Var}(\bar X)=\sigma^2/n \) — it **attains** the
bound and is efficient.

We verify by simulation: estimate \( \mathrm{Var}(\bar X) \) across many
replicates and compare to \( \sigma^2/n \).


In [ ]:
def normal_mean_cr_demo(mu_true, sigma2, n, M=200_000, seed=0):
    """Monte Carlo Var of the sample mean vs the CRLB sigma^2 / n."""
    rng = np.random.default_rng(seed)
    X = rng.normal(mu_true, np.sqrt(sigma2), size=(M, n))
    xbar = X.mean(axis=1)
    return xbar.var(ddof=0), sigma2 / n

mu_true = 3.0
sigma2 = 4.0
print(f"Normal(mu={mu_true}, sigma2={sigma2}):  I(mu)=1/sigma2={1/sigma2:.4f}, "
      f"CRLB = sigma2/n")
print(f"{'n':>5} {'MC Var(Xbar)':>14} {'CRLB sigma2/n':>16} {'ratio':>10}")
for n in [5, 10, 25, 50, 100]:
    var_mc, crlb = normal_mean_cr_demo(mu_true, sigma2, n, seed=n)
    print(f"{n:>5} {var_mc:>14.6f} {crlb:>16.6f} {var_mc/crlb:>10.4f}")

# Picture: Var(Xbar) vs n against the bound sigma2/n
ns = np.arange(5, 101, 5)
var_mc = [normal_mean_cr_demo(mu_true, sigma2, n, seed=n)[0] for n in ns]
plt.plot(ns, var_mc, "o-", label=r"MC $\mathrm{Var}(\bar X)$")
plt.plot(ns, sigma2 / ns, "k--", lw=2, label=r"CRLB $\sigma^2/n$")
plt.xlabel(r"$n$"); plt.ylabel("variance")
plt.title(f"Sample mean attains the Cramer-Rao bound (sigma2={sigma2})")
plt.legend(); plt.show()

print("Ratio MC Var / CRLB -> 1: the sample mean is efficient for the normal mean.")


---
## Part 4 — CR bound for Bernoulli and Poisson; is the MLE efficient?

For \( X_i \sim \mathrm{Bern}(p) \), \( I(p)=1/(p(1-p)) \), so the CRLB for an
unbiased estimator of \( p \) is \( p(1-p)/n \). The MLE \( \hat p = \bar X \)
has \( \mathrm{Var}=p(1-p)/n \), so it is **efficient**.

For \( X_i \sim \mathrm{Pois}(\lambda) \), \( I(\lambda)=1/\lambda \), so the
CRLB for an unbiased estimator of \( \lambda \) is \( \lambda/n \). The MLE
\( \hat\lambda=\bar X \) has \( \mathrm{Var}=\lambda/n \) — also efficient.

We confirm both by Monte Carlo: estimate the variance of \( \bar X \) from many
replicates and compare to \( 1/(n I(\theta)) \).


In [ ]:
def bernoulli_cr_demo(p_true, n, M=400_000, seed=0):
    rng = np.random.default_rng(seed)
    X = rng.binomial(1, p_true, size=(M, n))
    xbar = X.mean(axis=1)
    I_p = 1.0 / (p_true * (1 - p_true))
    return xbar.var(ddof=0), p_true * (1 - p_true) / n, I_p

def poisson_cr_demo(lam_true, n, M=400_000, seed=0):
    rng = np.random.default_rng(seed)
    X = rng.poisson(lam_true, size=(M, n))
    xbar = X.mean(axis=1)
    I_lam = 1.0 / lam_true
    return xbar.var(ddof=0), lam_true / n, I_lam

p_true = 0.3
print(f"Bernoulli p={p_true}: I(p)=1/(p(1-p))={1/(p_true*(1-p_true)):.4f}")
print(f"{'n':>5} {'MC Var(p_hat)':>14} {'CRLB p(1-p)/n':>16} {'ratio':>10}")
for n in [5, 10, 25, 50, 100]:
    v_mc, crlb, I_p = bernoulli_cr_demo(p_true, n, seed=n)
    print(f"{n:>5} {v_mc:>14.6f} {crlb:>16.6f} {v_mc/crlb:>10.4f}")

print()
lam_true = 2.0
print(f"Poisson lam={lam_true}: I(lam)=1/lam={1/lam_true:.4f}")
print(f"{'n':>5} {'MC Var(lam_hat)':>17} {'CRLB lam/n':>14} {'ratio':>10}")
for n in [5, 10, 25, 50, 100]:
    v_mc, crlb, I_lam = poisson_cr_demo(lam_true, n, seed=n)
    print(f"{n:>5} {v_mc:>17.6f} {crlb:>14.6f} {v_mc/crlb:>10.4f}")

# Plot: MC variance and CRLB as functions of n for Poisson
ns = np.arange(5, 101, 5)
v_mc_pois = [poisson_cr_demo(lam_true, n, seed=n)[0] for n in ns]
plt.plot(ns, v_mc_pois, "o-", label=r"MC $\mathrm{Var}(\hat\lambda)$")
plt.plot(ns, lam_true / ns, "k--", lw=2, label=r"CRLB $\lambda/n$")
plt.xlabel(r"$n$"); plt.ylabel("variance")
plt.title(f"Poisson MLE is efficient (lam={lam_true})")
plt.legend(); plt.show()

print("In both families the MLE = sample mean achieves the CRLB exactly:")
print("the ratio of Monte-Carlo variance to the bound converges to 1.")


---
## Your turn

- For \( X_i \sim \mathrm{Exp}(\lambda) \) (rate \( \lambda \)), derive
  \( I(\lambda) \) from the second-derivative formula and confirm it by Monte
  Carlo using the score-variance form. Is \( \hat\lambda = 1/\bar X \) unbiased?
  (It is not — so the CRLB does not apply directly; compute its actual variance and
  compare to \( 1/(n I(\lambda)) \).)
- Show that for \( X_i \sim N(\mu, \sigma^2) \) with \( \mu \) known, the
  Fisher information for \( \sigma^2 \) is \( 1/(2\sigma^4) \). Verify
  numerically that the unbiased estimator \( \hat\sigma^2 = \frac{1}{n}\sum (X_i-\mu)^2 \)
  attains the CRLB \( 2\sigma^4/n \).
- Verify the CRLB for the Bernoulli MLE with \( p=0.5 \) and \( n=200 \): is the
  ratio of empirical variance to \( p(1-p)/n \) closer to 1 than for \( p=0.05 \)?
  Why does the bound become harder to hit near the boundary?
- Compute the observed information \( -\frac{1}{n}\sum_i \ell''(\hat\theta;X_i) \)
  at the MLE for a Poisson sample and confirm it is close to \( n/\hat\lambda \)
  (so \( I_n(\hat\theta)\approx n\, I(\hat\theta) \)).


---
# Problem Set

**Try each problem before revealing the solution.**


## Problems

1. For \( X_1,\dots,X_n \stackrel{iid}{\sim} \mathrm{Bern}(p) \), derive the score \( U_p(X) \), the Fisher information \( I(p) \), and the Cramer-Rao lower bound for any unbiased estimator of \( p \). Show the MLE \( \hat p = \bar X \) attains the bound by computing \( \mathrm{Var}(\bar X) \). Verify the ratio \( \mathrm{Var}(\bar X) \big/ \mathrm{CRLB} \to 1 \) by simulation for \( n=10,50,200 \).

2. For \( X_1,\dots,X_n \stackrel{iid}{\sim} \mathrm{Pois}(\lambda) \), compute \( I(\lambda) \) two ways — via \( E[U_\lambda^2] \) and via \( -E[\partial^2 \log f/\partial\lambda^2] \) — and confirm both equal \( 1/\lambda \). Then verify by Monte Carlo that the observed information \( -\frac{1}{n}\sum_i \ell''(\hat\lambda;X_i) \) at the MLE \( \hat\lambda=\bar X \) is close to \( n/\hat\lambda \).

3. Let \( X_1,\dots,X_n \stackrel{iid}{\sim} N(\mu,\sigma^2) \) with \( \sigma^2 \) known. Derive \( I(\mu)=1/\sigma^2 \) from the log-likelihood, state the CRLB for an unbiased estimator of \( \mu \), and prove the sample mean is efficient by showing \( \mathrm{Var}(\bar X)=\sigma^2/n \). Confirm by simulation for \( \sigma^2=4 \) and \( n=5,25,100 \).

4. For \( X_1,\dots,X_n \stackrel{iid}{\sim} \mathrm{Exp}(\lambda) \) with rate \( \lambda \), compute \( I(\lambda)=1/\lambda^2 \) and state the CRLB for an unbiased estimator of \( \lambda \). Explain why the MLE \( \hat\lambda = 1/\bar X \) is biased and so the CRLB does not directly apply; estimate \( \mathrm{Var}(\hat\lambda) \) by simulation and compare it to \( 1/(n I(\lambda))=\lambda^2/n \).

5. Let \( X_1,\dots,X_n \stackrel{iid}{\sim} N(\mu,\sigma^2) \) with \( \mu \) known. Derive the Fisher information for \( \sigma^2 \), \( I(\sigma^2)=1/(2\sigma^4) \), and the CRLB \( 2\sigma^4/n \) for an unbiased estimator of \( \sigma^2 \). Show that \( \hat\sigma^2 = \frac{1}{n}\sum_{i=1}^n (X_i-\mu)^2 \) attains the bound and is therefore efficient. Verify by simulation for \( \mu=0 \), \( \sigma^2=9 \), and \( n=10,50,200 \).


<details>
<summary><strong>Reveal solutions</strong></summary>

**1.** For Bernoulli, \( \log f(x;p)=x\log p+(1-x)\log(1-p) \),
so \( U_p(x)=x/p-(1-x)/(1-p) \) and
\( \partial^2\log f/\partial p^2 = -x/p^2-(1-x)/(1-p)^2 \). Taking expectation
with \( E[X]=p \): \( I(p) = p/p^2 + (1-p)/(1-p)^2 = 1/(p(1-p)) \). The CRLB for
unbiased \( T \) estimating \( p \) is
\( p(1-p)/n \). Since \( \mathrm{Var}(\bar X)=p(1-p)/n \), the MLE attains the
bound and is efficient.

```python
import numpy as np
p_true = 0.3; M = 400_000
print(f"theory I(p) = {1/(p_true*(1-p_true)):.4f}, CRLB = p(1-p)/n")
for n in [10, 50, 200]:
    rng = np.random.default_rng(n)
    X = rng.binomial(1, p_true, size=(M, n))
    v = X.mean(axis=1).var()
    crlb = p_true * (1 - p_true) / n
    print(f"n={n:3d}  Var(p_hat)={v:.6f}  CRLB={crlb:.6f}  ratio={v/crlb:.4f}")
```
The ratio converges to 1 as \( n \) grows.

**2.** Poisson: \( \log f(x;\lambda)=-\lambda+x\log\lambda-\log(x!) \),
\( U_\lambda=-1+x/\lambda \), so
\( E[U^2]=\mathrm{Var}(X/\lambda)=\lambda/\lambda^2=1/\lambda \). Also
\( \partial^2\log f/\partial\lambda^2=-x/\lambda^2 \) and
\( -E[\cdot]=\lambda/\lambda^2=1/\lambda \). The observed information at the
MLE \( \hat\lambda=\bar X \) is
\( -\frac{1}{n}\sum_i (-X_i/\hat\lambda^2) = \bar X/\bar X^2 = 1/\bar X = n/\sum X_i \approx 1/\hat\lambda \),
and \( I_n(\hat\lambda)=n/\hat\lambda \).

```python
import numpy as np
lam_true = 2.0; n = 200; M = 50_000
rng = np.random.default_rng(0)
xbar = []
for _ in range(M):
    x = rng.poisson(lam_true, size=n)
    lam_hat = x.mean()
    xbar.append(-np.mean(x / lam_hat**2))   # observed info per obs at MLE
xbar = np.array(xbar)
print(f"mean observed info per obs = {xbar.mean():.5f}")
print(f"theory 1/lam_hat ~ 1/lam    = {1/lam_true:.5f}")
print(f"I_n(lam_hat) ~ n/lam_hat    = {n/lam_true:.5f}")
```

**3.** Normal mean: \( \log f(x;\mu)= -\frac12\log(2\pi\sigma^2)-(x-\mu)^2/(2\sigma^2) \),
\( U_\mu=(x-\mu)/\sigma^2 \), \( \partial^2\log f/\partial\mu^2=-1/\sigma^2 \),
so \( I(\mu)=1/\sigma^2 \) and \( I_n(\mu)=n/\sigma^2 \). The CRLB for
unbiased \( T \) of \( \mu \) is \( \sigma^2/n \). Since
\( \mathrm{Var}(\bar X)=\sigma^2/n \), \( \bar X \) attains the bound.

```python
import numpy as np
mu_true = 1.0; sigma2 = 4.0; M = 200_000
for n in [5, 25, 100]:
    rng = np.random.default_rng(n)
    X = rng.normal(mu_true, np.sqrt(sigma2), size=(M, n))
    v = X.mean(axis=1).var()
    crlb = sigma2 / n
    print(f"n={n:3d}  Var(Xbar)={v:.6f}  CRLB={crlb:.6f}  ratio={v/crlb:.4f}")
```

**4.** Exponential rate \( \lambda \): \( \log f(x;\lambda)=\log\lambda-\lambda x \),
\( U_\lambda=1/\lambda-x \), \( \partial^2\log f/\partial\lambda^2=-1/\lambda^2 \),
so \( I(\lambda)=1/\lambda^2 \). The CRLB for an unbiased estimator of \( \lambda \)
is \( 1/(n I(\lambda))=\lambda^2/n \). The MLE \( \hat\lambda=1/\bar X \) is
biased (\( E[1/\bar X]\ne \lambda \)), so the CRLB need not be attained.

```python
import numpy as np
lam_true = 2.0; M = 300_000
for n in [10, 50, 200]:
    rng = np.random.default_rng(n)
    X = rng.exponential(1.0 / lam_true, size=(M, n))
    lam_hat = 1.0 / X.mean(axis=1)
    v = lam_hat.var()
    crlb = lam_true**2 / n
    bias = lam_hat.mean() - lam_true
    print(f"n={n:3d}  bias={bias:.5f}  Var(lam_hat)={v:.6f}  CRLB={crlb:.6f}  ratio={v/crlb:.3f}")
```
\( \hat\lambda \) has positive bias and its variance exceeds the CRLB (which
applies only to unbiased estimators).

**5.** Normal variance with \( \mu \) known: write
\( \theta=\sigma^2 \). \( \log f(x;\theta)=-\frac12\log(2\pi\theta)-(x-\mu)^2/(2\theta) \).
\( \partial\log f/\partial\theta = -\frac{1}{2\theta}+\frac{(x-\mu)^2}{2\theta^2} \),
\( \partial^2\log f/\partial\theta^2 = \frac{1}{2\theta^2}-\frac{(x-\mu)^2}{\theta^3} \).
Taking \( E[(X-\mu)^2]=\sigma^2=\theta \):
\( I(\theta)= -E[\partial^2\log f/\partial\theta^2]
= -\frac{1}{2\theta^2}+\frac{\theta}{\theta^3} = \frac{1}{2\theta^2} = \frac{1}{2\sigma^4} \).
The CRLB for an unbiased estimator of \( \sigma^2 \) is
\( 1/(n I(\sigma^2))=2\sigma^4/n \). Now
\( \hat\sigma^2=\frac{1}{n}\sum (X_i-\mu)^2 \) has
\( n\hat\sigma^2/\sigma^2\sim\chi^2_n \), so
\( \mathrm{Var}(\hat\sigma^2)=2\sigma^4/n \) — it attains the bound and is
efficient.

```python
import numpy as np
mu = 0.0; sigma2 = 9.0; M = 300_000
for n in [10, 50, 200]:
    rng = np.random.default_rng(n)
    X = rng.normal(mu, np.sqrt(sigma2), size=(M, n))
    s2 = ((X - mu) ** 2).mean(axis=1)
    v = s2.var()
    crlb = 2 * sigma2**2 / n
    print(f"n={n:3d}  Var(s2_hat)={v:.6f}  CRLB={crlb:.6f}  ratio={v/crlb:.4f}")
```
The ratio is approximately 1, confirming efficiency.


</details>
